# customer-data-cleaning-pipeline

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_excel("Customer Call List.xlsx")

In [3]:
df.head()

,CustomerID,First_Name,Last_Name,Phone_Number,Address,Paying Customer,Do_Not_Contact,Not_Useful_Column
0,1001,Frodo,Baggins,123-545-5421,"123 Shire Lane, Shire",Yes,No,True
1,1002,Abed,Nadir,123/643/9775,93 West Main Street,No,Yes,False
2,1003,Walter,/White,7066950392,298 Drugs Driveway,N,NaN,True
3,1004,Dwight,Schrute,123-543-2345,"980 Paper Avenue, Pennsylvania, 18503",Yes,Y,True
4,1005,Jon,Snow,876|678|3469,123 Dragons Road,Y,No,True


In [4]:
df.drop_duplicates(inplace = True)                      # remove dupliactes

In [5]:
df.drop(columns='Not_Useful_Column', inplace=True)      # remove 'Not_Useful_Column'  column

In [6]:
df["First_Name"] = df["First_Name"].str.strip("123._/") # remove extra literals in firstnames

In [7]:
df["Last_Name"] = df["Last_Name"].str.strip("123._/")   # remove extra literals in lasstnames

In [8]:
df["Phone_Number"] = df["Phone_Number"].str.replace('[^a-zA-Z0-9]', '', regex=True) # Standardize Phone_Numbers

In [9]:
df["Phone_Number"] = df["Phone_Number"].astype(str)  # Convert the Phone_Number column to string type to allow text manipulation
df["Phone_Number"] = (                               # Reformat phone numbers by inserting hyphens after the 3rd and 6th digits
    df["Phone_Number"].str[:3] + "-" +
    df["Phone_Number"].str[3:6] + "-" +
    df["Phone_Number"].str[6:] )

In [10]:
# Replace placeholder strings like 'Na--' or 'na--' with true missing value representations (np.nan)
df["Phone_Number"] = df["Phone_Number"].replace( r'^(Na--|na--)$', np.nan, regex=True) 

In [11]:
# Split the Address column by the first two commas into three separate columns

split_cols = df["Address"].str.split(",", n=2, expand=True)  # Standardize Address column

# Assign the split parts to new specific columns and remove any extra whitespace spaces
df["street"] = split_cols[0].str.strip()
df["state"] = split_cols[1].str.strip()
df["zipcode"] = split_cols[2].str.strip()

In [12]:
df.drop(columns='Address' , inplace= True)                     # remove old address column

In [13]:
df.head()

,CustomerID,First_Name,Last_Name,Phone_Number,Paying Customer,Do_Not_Contact,street,state,zipcode
0,1001,Frodo,Baggins,123-545-5421,Yes,No,123 Shire Lane,Shire,NaN
1,1002,Abed,Nadir,123-643-9775,No,Yes,93 West Main Street,NaN,NaN
2,1003,Walter,White,NaN,N,NaN,298 Drugs Driveway,NaN,NaN
3,1004,Dwight,Schrute,123-543-2345,Yes,Y,980 Paper Avenue,Pennsylvania,18503
4,1005,Jon,Snow,876-678-3469,Y,No,123 Dragons Road,NaN,NaN


In [14]:
# Make all text lowercase
df['Paying Customer'] = df['Paying Customer'].str.lower()  # standardize "paying customer" column

# Group and change all positive variations to 'Yes'
df['Paying Customer'] = df['Paying Customer'].replace(['yes', 'y', 'Yes'], 'Yes')

# Group and change all negative variations to 'No'
df['Paying Customer'] = df['Paying Customer'].replace(['no', 'n', 'n/a'], 'No')

In [15]:
# View the cleaned target column
df[['Paying Customer']]

,Paying Customer
0,Yes
1,No
2,No
3,Yes
4,Yes
5,Yes
6,No
7,No
8,Yes
9,Yes


In [16]:
df['Do_Not_Contact'] = df['Do_Not_Contact'].str.lower()    # Standardize "Do_Not_Contact" column
df['Do_Not_Contact'] = df['Do_Not_Contact'].replace(['yes', 'y'], 'yes')
df['Do_Not_Contact'] = df['Do_Not_Contact'].replace(['no', 'n'], 'no')

In [17]:
df[['Do_Not_Contact']]

,Do_Not_Contact
0,no
1,yes
2,NaN
3,yes
4,no
5,yes
6,no
7,no
8,NaN
9,no


In [18]:
# Replace actual NaN values with empty strings
df = df.fillna('')

In [19]:
# Replace any 'N/a' or 'n/a' strings with empty strings
df = df.replace(['N/a', 'n/a'], '', regex=False)

In [20]:
df[['Do_Not_Contact']]

,Do_Not_Contact
0,no
1,yes
2,
3,yes
4,no
5,yes
6,no
7,no
8,
9,no


In [21]:
df = df[df['Do_Not_Contact'] != 'yes']      # remove customers not to contact

In [22]:
df = df.reset_index(drop=True)

In [23]:
df['Do_Not_Contact'] = df['Do_Not_Contact'].replace('', 'no')  # fill empty_value with "no"

In [24]:
df[['Do_Not_Contact']]

,Do_Not_Contact
0,no
1,no
2,no
3,no
4,no
5,no
6,no
7,no
8,no
9,no


In [25]:
df = df[df['Phone_Number'].str.strip() != '']  # remove rows with no phoneNumbers

In [26]:
df = df.reset_index(drop=True)

In [27]:
df.to_csv("Customer_Call_List_Cleaned.csv", index=False)